# Fine-tune Llama-3.1-8B for coffee-shop message routing (QLoRA via Unsloth)

**Status: authored, NOT executed.** I have no funded GPU/API access in the environment this was written in. This notebook is ready to run as-is on a free Google Colab T4 or a GCP T4/L4 instance -- per Unsloth's own published benchmarks, a run this size (a few hundred short examples, short structured-JSON outputs) should complete in well under an hour and cost roughly $0-5 in GPU time (free on Colab's T4 tier).

**Why fine-tune this specific task**: `RouterAgent`'s job -- classify a coffee-shop message and pick the right downstream agent -- is a narrow, well-specified classification task with a small, fixed output schema. Per the research behind this round (see the main README), this is exactly the kind of task where a small QLoRA adapter on an 8B model can be pushed toward near-deterministic structured output and lower per-request latency (shorter prompts, no few-shot examples needed in-context), rather than a case for prompting alone.

**Before treating any result from this notebook as a real improvement**: run `../api/eval_harness.py` (or the eval logic adapted below) against BOTH the base model and the fine-tuned adapter, and report the actual before/after accuracy, JSON-schema-validity rate, and latency numbers. Do not claim an improvement without that comparison.

## 1. Install dependencies (Colab / GCP T4-L4)

In [ ]:
# NOT RUN -- adapted from Unsloth's official Llama-3.1 (8B)-Alpaca.ipynb recipe.
!pip install --upgrade pip -q
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps xformers trl peft accelerate bitsandbytes -q

## 2. Generate the training dataset

Run `generate_training_data.py` (in this same directory) first, or from a cell here -- this script is pure Python with no external dependencies and IS runnable/verified (see the repo's test suite output for confirmation this script and its inputs work as expected).

In [ ]:
# This cell CAN actually be run (no GPU/API key needed) -- it just wasn't run as
# part of this PR to keep the notebook's own execution count honest (nothing in
# this notebook has been executed).
!python generate_training_data.py router_training_data.jsonl

## 3. Load the base model with 4-bit quantization + attach a LoRA adapter

Rank 32 applied to all linear layers (q/k/v/o + gate/up/down projections), matching Unsloth's documented recipe for this model size.

In [ ]:
# NOT RUN.
from unsloth import FastLanguageModel
import torch

max_seq_length = 512  # short prompts + short JSON outputs -- no need for a large context here

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,  # auto-detect (bfloat16 on Ampere+, float16 otherwise)
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 4. Load and format the dataset (Alpaca-style prompt template)

In [ ]:
# NOT RUN.
from datasets import load_dataset

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def format_example(example):
    text = ALPACA_PROMPT.format(example["instruction"], example["input"], example["output"]) + EOS_TOKEN
    return {"text": text}

dataset = load_dataset("json", data_files="router_training_data.jsonl", split="train")
dataset = dataset.map(format_example)

## 5. Train

In [ ]:
# NOT RUN. 3 epochs over ~240 short examples should complete in minutes on a T4,
# per Unsloth's published benchmarks for datasets of this size/shape.
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        output_dir="router_lora_outputs",
    ),
)

trainer_stats = trainer.train()

## 6. Save the adapter

In [ ]:
# NOT RUN.
model.save_pretrained("router_lora_adapter")
tokenizer.save_pretrained("router_lora_adapter")

## 7. Evaluate BEFORE vs. AFTER -- do not skip this step

Run `../api/eval_harness.py`'s `run_eval()` (import it directly, or adapt inline) against:
1. The base model (no adapter loaded) -- this is your baseline.
2. The fine-tuned adapter loaded via `FastLanguageModel.from_pretrained(..., adapter="router_lora_adapter")` (or merged weights, per Unsloth's inference docs).

Report both numbers side by side (accuracy, JSON-schema-validity rate, latency) in the README. A fine-tune without this comparison is not evidence of an improvement -- see the honesty note at the top of this notebook.

In [ ]:
# NOT RUN -- sketch only; adapt to however you're serving the fine-tuned model
# (a local Unsloth inference call, or exporting to a vLLM-servable format and
# reusing llm_client.py against that endpoint).
import sys
sys.path.insert(0, "../api")
from eval_harness import run_eval, print_report
from eval_dataset import EXAMPLES

# router_base = RouterAgent(...)       # pointed at the base model
# router_finetuned = RouterAgent(...)  # pointed at the fine-tuned adapter
# print_report(run_eval(router_base, EXAMPLES), provider="base")
# print_report(run_eval(router_finetuned, EXAMPLES), provider="finetuned")